# ETRS923 – TP3 – Projet AMR & EJUB : classification météo au Bourget-du-Lac
MeteoStat → TensorFlow → ONNX (MATLAB / ThingSpeak) + fichier C pour le STM32N6.

**Entrées du réseau : uniquement les mesures de la carte** (pas de coordonnées GPS, pas de date ni d'heure).

| Entrée | D'où ça vient sur la carte |
|---|---|
| `temp` (°C), `rhum` (%) | capteur SHT40 |
| `pres` (hPa, niveau de la mer) | capteur de pression, converti avec l'altitude du Bourget-du-Lac (240 m) |
| `spread` (°C) | température − point de rosée (calculé à partir de temp et rhum) |
| `dpres1h`, `dpres3h`, `dpres6h` | évolution de la pression sur 1, 3, 6 h (historique gardé sur la carte) |
| `dtemp1h`, `dtemp3h`, `drhum1h`, `drhum3h` | évolution de la température et de l'humidité sur 1 et 3 h |

Le notebook compare **3 jeux d'entrées** (mesures instantanées seules, + tendance 3 h, + toutes les tendances)
et garde automatiquement le meilleur. **Sorties : 4 classes → Soleil, Nuageux, Pluie, Neige.**

Exécution : **Exécution → Tout exécuter** (≈ 30 min). Les fichiers se téléchargent à la fin.

In [ ]:
!pip -q install "meteostat<2" tf2onnx onnx onnxruntime

In [ ]:
import numpy as np, pandas as pd, json, time
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)
print("TensorFlow", tf.__version__)

## 1. Données MeteoStat : Le Bourget-du-Lac (+ 3 villes voisines au climat proche pour avoir plus d'exemples)
Le champ `coco` (code météo 1 à 27) sert d'étiquette. La position n'est **pas** une entrée du réseau :
les villes voisines servent seulement à avoir plus d'heures d'apprentissage.

In [ ]:
from meteostat import Point, Hourly
from datetime import datetime

LOCS = {                                    # (latitude, longitude, altitude en m)
    "Le Bourget-du-Lac": (45.65, 5.86, 240),
    "Annecy":            (45.90, 6.12, 448),
    "Grenoble":          (45.19, 5.73, 214),
    "Lyon":              (45.76, 4.84, 200),
}
start, end = datetime(2015, 1, 1), datetime(2024, 12, 31, 23)

frames = []
for name, (lat, lon, alt) in LOCS.items():
    d = Hourly(Point(lat, lon, alt), start, end).fetch()
    if d.empty:
        print(f"{name:18s} aucune donnée, ville ignorée"); continue
    d = d.asfreq("h")                                  # une ligne par heure, trous = NaN
    for h in (1, 3, 6):                                # tendances (valeur(t) - valeur(t - h))
        d[f"dpres{h}h"] = d["pres"] - d["pres"].shift(h)
    for h in (1, 3):
        d[f"dtemp{h}h"] = d["temp"] - d["temp"].shift(h)
        d[f"drhum{h}h"] = d["rhum"] - d["rhum"].shift(h)
    d["ville"] = name
    frames.append(d)
    print(f"{name:18s} {len(d):6d} heures, coco dispo : {d['coco'].notna().mean():.0%}")
raw = pd.concat(frames)
raw[["temp", "rhum", "pres", "coco"]] = raw[["temp", "rhum", "pres", "coco"]].astype(float)

## 2. Les 5 entrées (mêmes formules dans MATLAB et dans le code C de la carte)

In [ ]:
def dewpoint(t, rh):
    a, b = 17.62, 243.12                               # formule de Magnus
    g = np.log(np.clip(rh, 1, 100) / 100) + a * t / (b + t)
    return b * g / (a - g)

def to_sealevel(p_station, t, alt):
    return p_station * (1 - 0.0065 * alt / (t + 0.0065 * alt + 273.15)) ** -5.257

d = raw.dropna(subset=["temp", "rhum", "pres", "coco"]).copy()
d["spread"] = d["temp"] - dewpoint(d["temp"], d["rhum"])
d["coco"] = d["coco"].astype(int)
df = d

JEUX = {
    "instantane (4)":     ["temp", "rhum", "pres", "spread"],
    "+ tendance 3h (5)":  ["temp", "rhum", "pres", "dpres3h", "spread"],
    "+ tendances (11)":   ["temp", "rhum", "pres", "spread", "dpres1h", "dpres3h", "dpres6h",
                           "dtemp1h", "dtemp3h", "drhum1h", "drhum3h"],
}
TOUTES = sorted({f for fs in JEUX.values() for f in fs})
print(len(df), "heures exploitables")

## 3. Les 4 classes : Soleil, Nuageux, Pluie, Neige
| Classe | Codes coco de MeteoStat |
|---|---|
| **Soleil** | 1 clair, 2 beau temps |
| **Nuageux** | 3 nuageux, 4 couvert, 5-6 brouillard |
| **Pluie** | 7-11 pluie, 17-18 averses, 23-27 orage / grêle |
| **Neige** | 12-16 neige et pluie+neige, 19-22 averses de neige |

Test = année **2024**, entraînement = 2015-2023 : le modèle est évalué sur une année qu'il n'a jamais vue.

In [ ]:
CLASSES = {
    "Soleil":  [1, 2],
    "Nuageux": [3, 4, 5, 6],
    "Pluie":   [7, 8, 9, 10, 11, 17, 18, 23, 24, 25, 26, 27],
    "Neige":   [12, 13, 14, 15, 16, 19, 20, 21, 22],
}

# Mêmes heures pour tous les jeux d'entrées (comparaison équitable)
code2name = {c: n for n, cs in CLASSES.items() for c in cs}
dd = df[df["coco"].isin(code2name)].dropna(subset=TOUTES).copy()
dd["label"] = dd["coco"].map(code2name)
names = sorted(dd["label"].unique())
y = dd["label"].map({n: i for i, n in enumerate(names)}).values
test = np.asarray(dd.index.year == 2024)
def entrees(features):
    return dd[list(features)].values.astype("float32")
print(f"{len(dd)} heures (train {(~test).sum()} / test {test.sum()})")
print("Répartition du test 2024 :", dict(zip(names, np.bincount(y[test], minlength=len(names)))))

## 4. Recherche du meilleur réglage
On compare les 3 jeux d'entrées × 2 tailles de réseau × avec ou sans **équilibrage des classes** (les classes rares comptent plus à l'apprentissage :
meilleure détection de la neige, mais plus d'erreurs sur les classes fréquentes).
- *accuracy* = % de bonnes réponses sur 2024 ; *f1_macro* = qualité moyenne sur chaque classe.

In [ ]:
def build_model(n_in, n_out, units, Xtrain):
    norm = keras.layers.Normalization()
    norm.adapt(Xtrain)                                  # moyenne / écart-type du train
    layers = [keras.layers.Input((n_in,)), norm]
    for u in units:
        layers.append(keras.layers.Dense(u, activation="relu"))
    layers.append(keras.layers.Dense(n_out, activation="softmax"))
    return keras.Sequential(layers)

def run(jeu, units, equilibrage, epochs=60, lr=1e-3, batch=512):
    t0 = time.perf_counter()
    X = entrees(JEUX[jeu])
    Xtr, ytr, Xte, yte = X[~test], y[~test], X[test], y[test]
    p = np.random.RandomState(SEED).permutation(len(Xtr)); Xtr, ytr = Xtr[p], ytr[p]
    cw = None
    if equilibrage:
        cl = np.unique(ytr)
        cw = dict(zip(cl, compute_class_weight("balanced", classes=cl, y=ytr)))
    tf.random.set_seed(SEED)
    m = build_model(X.shape[1], len(names), units, Xtr)
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    es = keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor="val_loss")
    h = m.fit(Xtr, ytr, validation_split=0.15, epochs=epochs, batch_size=batch,
              class_weight=cw, callbacks=[es], verbose=0)
    pred = m.predict(Xte, verbose=0).argmax(1)
    info = dict(entrees=jeu, archi=str(units), equilibrage=equilibrage,
                epochs=len(h.history["loss"]), params=m.count_params(),
                taille_ko=round(m.count_params() * 4 / 1024, 1),
                accuracy=round(100 * (pred == yte).mean(), 1),
                f1_macro=round(100 * f1_score(yte, pred, average="macro"), 1),
                hasard=round(100 / len(names), 1),
                duree_s=round(time.perf_counter() - t0))
    return m, info

essais, modeles = [], {}
for jeu in JEUX:
    for a in [(64, 32), (128, 64)]:
        for eq in [False, True]:
            m, info = run(jeu, a, eq)
            modeles[len(essais)] = m
            essais.append(info)
            print(info)

resultats = pd.DataFrame(essais)
resultats.sort_values("accuracy", ascending=False)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
lab = [f"{r.entrees}\n{r.archi}\n{'équil.' if r.equilibrage else 'non équil.'}" for r in resultats.itertuples()]
ax.bar(range(len(resultats)), resultats["accuracy"], color="tab:blue", label="accuracy")
ax.plot(range(len(resultats)), resultats["f1_macro"], "o", color="tab:orange", label="f1 macro")
ax.set_xticks(range(len(resultats))); ax.set_xticklabels(lab, fontsize=7)
ax.set_ylabel("% (test 2024)"); ax.set_ylim(0, 100); ax.grid(axis="y", alpha=.3); ax.legend()
ax.set_title("Comparaison des réglages (4 classes, entrées capteurs uniquement)")
plt.tight_layout(); plt.show()

## 5. Modèle retenu : le réglage avec la meilleure accuracy

In [ ]:
choix = resultats["accuracy"].idxmax()
model, r = modeles[choix], essais[choix]
FEATURES = JEUX[r["entrees"]]
X = entrees(FEATURES)
Xte, yte = X[test], y[test]
print("MODELE RETENU :", r)
print("Entrées :", FEATURES)
print("\nMeilleure accuracy par jeu d'entrées :")
print(resultats.groupby("entrees")["accuracy"].max().to_string())
print(f"\n=> {r['accuracy']} % de bonnes réponses sur l'année 2024 ({len(yte)} heures jamais vues), "
      f"contre {r['hasard']} % au hasard.")

pred = model.predict(Xte, verbose=0).argmax(1)
print("\nTaux de réussite par classe :")
for i, n in enumerate(names):
    sel = yte == i
    print(f"  {n:8s} {100 * (pred[sel] == i).mean():5.1f} %   ({sel.sum()} heures)")
ConfusionMatrixDisplay(confusion_matrix(yte, pred, labels=range(len(names))),
                       display_labels=names).plot()
plt.title("Matrice de confusion (test 2024)"); plt.show()

## 6. Export TensorFlow → ONNX
La normalisation est fusionnée dans la 1re couche Dense : le réseau exporté ne contient plus que Dense / ReLU / Softmax.

In [ ]:
import tf2onnx, onnx, onnxruntime as ort

N_FEAT = len(FEATURES)
norm = model.layers[0]
z = np.zeros((1, N_FEAT), "float32"); o = np.ones((1, N_FEAT), "float32")
shift = np.array(norm(z))[0]
scale = np.array(norm(o))[0] - shift
dense = [l for l in model.layers if isinstance(l, keras.layers.Dense)]
export = keras.Sequential([keras.layers.Input((N_FEAT,))] +
                          [keras.layers.Dense(l.units, activation=l.activation) for l in dense])
couches = []                                            # (W [entrees, sorties], b) pour le code C
for i, (le, l) in enumerate(zip(export.layers, dense)):
    W, b = l.get_weights()
    if i == 0:
        b = b + shift @ W
        W = W * scale[:, None]
    le.set_weights([W, b])
    couches.append((W.astype("float32"), b.astype("float32")))
print("Écart max modèle / modèle sans normalisation :",
      np.abs(export.predict(Xte[:200], verbose=0) - model.predict(Xte[:200], verbose=0)).max())

spec = (tf.TensorSpec((None, N_FEAT), tf.float32, name="input"),)

@tf.function(input_signature=spec)
def serve(x):
    return export(x)

model.save("meteo_model.keras")
onnx_model, _ = tf2onnx.convert.from_function(serve, input_signature=spec, opset=13, output_path="meteo.onnx")
assert len(onnx_model.graph.input) == 1, "L'ONNX a des entrées en trop"

sess = ort.InferenceSession("meteo.onnx")
iname = sess.get_inputs()[0].name
out_onnx = sess.run(None, {iname: Xte})[0]
print("Écart max Keras/ONNX :", np.abs(out_onnx[:200] - model.predict(Xte[:200], verbose=0)).max())
print("Accuracy de l'ONNX sur 2024 : %.1f %%" % (100 * (out_onnx.argmax(1) == yte).mean()))

t = time.perf_counter()
for _ in range(1000): sess.run(None, {iname: Xte[:1]})
print("Inférence CPU Colab : %.3f ms / échantillon" % (time.perf_counter() - t))

## 7. Fichier C pour la carte : `meteo_modele.h`
Sur la carte : remplacer `Appli/Core/Inc/meteo_modele.h` par ce fichier, puis Clean + Build. Rien d'autre à modifier.

In [ ]:
def _f(v):
    s = "%.9g" % float(v)
    if "." not in s and "e" not in s and "n" not in s:
        s += ".0"
    return s + "f"

def _tab(v, par_ligne=8):
    v = np.asarray(v).ravel()
    return ",\n".join("  " + ", ".join(_f(x) for x in v[i:i + par_ligne])
                      for i in range(0, len(v), par_ligne))

def ecrire_entete_c(chemin, couches, features, noms_sorties, description, tests,
                    classes=None, regroupement=None):
    """couches : [(W [n_in, n_out], b)] ; tests : [(entrees, texte attendu)].
    classes / regroupement : pour regrouper plusieurs sorties du reseau dans une meme classe
    (par defaut : une classe par sortie)."""
    classes = classes or noms_sorties
    regroupement = regroupement or list(range(len(noms_sorties)))
    tailles = [couches[0][0].shape[0]] + [W.shape[1] for W, _ in couches]
    assert tailles[0] == len(features) and tailles[-1] == len(noms_sorties)
    besoin_heure = int(any(f.startswith(("doy", "hour")) for f in features))
    heures_hist = max([int(f[-2]) for f in features if f.startswith("d") and f.endswith("h")] or [0])
    L = ["/**",
         " * meteo_modele.h - ETRS923 TP3, projet AMR & EJUB : reseau meteo pour le STM32N6",
         " * FICHIER GENERE AUTOMATIQUEMENT par le notebook Colab (section 7) : ne pas modifier a la main.",
         " * " + description,
         " * Entrees : " + ", ".join(features),
         " * Architecture : " + " -> ".join(map(str, tailles)) + " (ReLU puis Softmax), poids W[entree][sortie].",
         " */",
         "#ifndef METEO_MODELE_H", "#define METEO_MODELE_H", "",
         '#define METEO_MODELE_DESCRIPTION "%s"' % description,
         "#define METEO_NB_ENTREES        %d" % tailles[0],
         "#define METEO_LISTE_ENTREES     { %s }" % ", ".join("METEO_E_" + f.upper() for f in features),
         "#define METEO_BESOIN_HEURE      %d   /* 1 si le reseau utilise la date / l'heure */" % besoin_heure,
         "#define METEO_HISTORIQUE_H      %d   /* heures d'historique necessaires aux tendances */" % heures_hist,
         "#define METEO_NB_SORTIES_RESEAU %d" % tailles[-1],
         "#define METEO_NB_CLASSES        %d" % len(classes),
         "#define METEO_NOMS_CLASSES      { %s }" % ", ".join('"%s"' % n for n in classes),
         "#define METEO_REGROUPEMENT      { %s }   /* sortie du reseau -> classe */" % ", ".join(map(str, regroupement)),
         "#define METEO_NB_COUCHES        %d" % len(couches),
         "#define METEO_LARGEUR_MAX       %d" % max(tailles), ""]
    for k, (x, attendu) in enumerate(tests, 1):
        L.append("#define METEO_TEST%d_ENTREES { %s }" % (k, ", ".join(_f(t) for t in x)))
        L.append('#define METEO_TEST%d_ATTENDU "%s"' % (k, attendu))
    L += ["", "#ifdef METEO_MODELE_POIDS   /* defini uniquement dans meteo_reseau.c */",
          "static const int METEO_TAILLES[METEO_NB_COUCHES + 1] = { %s };" % ", ".join(map(str, tailles))]
    for i, (W, b) in enumerate(couches):
        L.append("static const float METEO_W%d[%d] = {\n%s\n};" % (i, W.size, _tab(W)))
        L.append("static const float METEO_B%d[%d] = {\n%s\n};" % (i, b.size, _tab(b)))
    L.append("static const float *const METEO_W[METEO_NB_COUCHES] = { %s };"
             % ", ".join("METEO_W%d" % i for i in range(len(couches))))
    L.append("static const float *const METEO_B[METEO_NB_COUCHES] = { %s };"
             % ", ".join("METEO_B%d" % i for i in range(len(couches))))
    L += ["#endif /* METEO_MODELE_POIDS */", "", "#endif /* METEO_MODELE_H */", ""]
    with open(chemin, "w") as fo:
        fo.write("\n".join(L))

# Deux échantillons de test de classes prédites différentes, avec le résultat ONNX attendu
k1 = 0
k2 = int(np.argmax(out_onnx.argmax(1) != out_onnx[k1].argmax()))
tests = []
for k in (k1, k2):
    c = int(out_onnx[k].argmax())
    tests.append((Xte[k], "%s %.1f %%" % (names[c], 100 * out_onnx[k, c])))
    print(f"Test {len(tests)} : ligne {k + 1} de echantillons_test.csv -> attendu {tests[-1][1]} (vrai : {names[yte[k]]})")

description = "4 classes, capteurs seuls %s, reseau %s, %s, accuracy test 2024 = %.1f %%" % (
    r["entrees"], r["archi"], "classes equilibrees" if r["equilibrage"] else "sans equilibrage", r["accuracy"])
ecrire_entete_c("meteo_modele.h", couches, FEATURES, names, description, tests)
print("\nmeteo_modele.h écrit :", description)

## 8. Fichiers pour MATLAB / la carte + téléchargement

In [ ]:
with open("labels.json", "w") as fo: json.dump(names, fo, ensure_ascii=False)
with open("features.json", "w") as fo:
    json.dump({"features": FEATURES, "classes": names,
               "lieu": {"nom": "Le Bourget-du-Lac", "alt": 240},
               "notes": "pres = pression ramenee au niveau de la mer (alt 240 m) ; dXXXNh = XXX(t) - XXX(t - N h)"},
              fo, ensure_ascii=False, indent=1)
pd.DataFrame(Xte[:50], columns=FEATURES).assign(label_vrai=[names[i] for i in yte[:50]]).to_csv(
    "echantillons_test.csv", index=False)
resultats.to_csv("resultats_reglages.csv", index=False)

print("Liste des classes pour le script MATLAB (dans cet ordre) :")
print("classes = [" + ", ".join('"%s"' % n for n in names) + "];")

from google.colab import files
for fn in ["meteo.onnx", "meteo_modele.h", "labels.json", "features.json",
           "echantillons_test.csv", "resultats_reglages.csv", "meteo_model.keras"]:
    files.download(fn)